# Load data

In [1]:
#required libraries and paths
import os
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Colab Notebooks/Imperial ML/Capstone/initial_data"

ValueError: mount failed

In [ ]:
# LOAD Existing data and append as needed

function_data = {}

for n in range(1, 9):
    # read in updated data if it exists
    func_path = f"{base_path}/function_{n}"
    x_updated_path = f"{func_path}/updated_inputs.npy"
    y_updated_path = f"{func_path}/updated_outputs.npy"

    if os.path.exists(x_updated_path) and os.path.exists(y_updated_path):
        inputs = np.load(x_updated_path)
        outputs = np.load(y_updated_path)
    else:
        inputs = np.load(f"{func_path}/initial_inputs.npy")
        outputs = np.load(f"{func_path}/initial_outputs.npy")

    # Add this week's single new point
    #x_new = X_new[n - 1].reshape(1, -1)
    #y_new_val = np.array([y_new[n - 1]])

    #X_updated = np.vstack([inputs, x_new])
    #y_updated = np.concatenate([outputs, y_new_val])

    # Save clean updated version
    #np.save(f"{func_path}/updated_inputs.npy", X_updated)
    #np.save(f"{func_path}/updated_outputs.npy", y_updated)

    X_updated = np.vstack([inputs])
    y_updated = np.concatenate([outputs])

    # DataFrame version
    columns = [f"x{i+1}" for i in range(X_updated.shape[1])]
    df = pd.DataFrame(X_updated, columns=columns)
    df["output"] = y_updated

    function_data[f"function_{n}"] = df

print(function_data["function_1"]) # week 4 results will be row 13 (starts at 0)

# Review performance

In [ ]:
# check if last query was new best?
# compare last datapoint
import numpy as np
import pandas as pd

bo_check = []

for n in range(1, 9):

    df = function_data[f"function_{n}"]

    old_y = df["output"].values

    # newest point = last row (since you appended it)
    new_y = old_y[-1]

    history_y = old_y[:-1]

    old_best = np.max(history_y)
    old_worst = np.min(history_y)
    old_mean = np.mean(history_y)
    old_std = np.std(history_y)

    z_score = (new_y - old_mean) / (old_std + 1e-8)

    is_new_best = new_y > old_best
    improvement = new_y - old_best

    percentile = (history_y < new_y).mean() * 100

    bo_check.append({
        "function": n,
        "new_output": new_y,
        "old_best": old_best,
        "improvement_vs_best": improvement,
        "is_new_best": is_new_best,
        "old_mean": old_mean,
        "old_std": old_std,
        "z_score": z_score,
        "percentile_vs_old": percentile,
        "old_worst": old_worst
    })

bo_check_df = pd.DataFrame(bo_check)

display(bo_check_df.sort_values("percentile_vs_old", ascending=False))



## Best so far curves

In [ ]:
# or best so far curves:

# Improvement over time:

for n in range(1, 9):
  df = function_data[f"function_{n}"]
  y = df["output"].values
  best_curve = np.maximum.accumulate(y)
  split = len(y)

  plt.figure(figsize=(7, 4))

  # Plot best_curve against 1-based observation numbers
  plt.plot(range(1, len(y) + 1), best_curve, marker="o")

  # The vertical line now correctly marks the last observation (the BO update point)
  plt.axvline(split, linestyle="--", label="BO update point")

  plt.title(f"Function {n}: Best-so-far curve")
  plt.xlabel("Observation Number") # Changed label for clarity
  plt.ylabel("Best-so-far output")

  plt.legend()
  plt.show()

# Plot Y values over time

In [ ]:
# plot y values across functions

import matplotlib.pyplot as plt
import pandas as pd

for n in range(1, 9):
    df = function_data[f"function_{n}"]
    y_values = df["output"]

    plt.figure(figsize=(8, 4))
    plt.plot(y_values, marker='o', linestyle='-')
    plt.title(f"Function {n}: Output Values Over Observations")
    plt.xlabel("Observation Index")
    plt.ylabel("Output Value")
    plt.grid(True)
    plt.show()

In [ ]:
# or distributions

# Where best sits in distribution:
# VIEW DISTRIBUTIONS
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

for n in range(1, 9):
    df = function_data[f"function_{n}"]

    y_old = df["output"].values[:-1]
    y_new = df["output"].values[-1]
    old_best = np.max(y_old)

    plt.figure(figsize=(6, 4))

    # KDE of old distribution
    sns.kdeplot(y_old, fill=True, alpha=0.4)

    # New point
    plt.axvline(y_new, linestyle="--", linewidth=2, label="New point")

    # Old best
    plt.axvline(old_best, linestyle=":", linewidth=2, label="Old best")

    plt.title(f"Function {n}: New sample vs distribution (KDE)")
    plt.xlabel("Output")
    plt.ylabel("Density")
    plt.legend()
    plt.show()


# Look at some clusters (use PCA for higher dimensions)

In [ ]:
# ============================================================
# CLUSTERING AND VISUALISATION PIPELINE FOR FUNCTIONS 1–8
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances


# ============================================================
# 1. DATA EXTRACTION AND VALIDATION
# ============================================================

def extract_function_data(df):
    """
    Extract X, y, and ordered input-column names.

    Expected DataFrame format:
        x1, x2, ..., output
    """
    if "output" not in df.columns:
        raise ValueError(
            "Each DataFrame must contain a column called 'output'."
        )

    input_columns = [
        col
        for col in df.columns
        if col != "output"
    ]

    # Sort x1, x2, ..., x8 numerically.
    input_columns = sorted(
        input_columns,
        key=lambda col: int(col[1:])
        if col.startswith("x") and col[1:].isdigit()
        else col,
    )

    X = df[input_columns].to_numpy(dtype=float)
    y = df["output"].to_numpy(dtype=float)

    if np.any(~np.isfinite(X)):
        raise ValueError("Inputs contain NaN or infinite values.")

    if np.any(~np.isfinite(y)):
        raise ValueError("Outputs contain NaN or infinite values.")

    if np.any((X < 0) | (X > 1)):
        raise ValueError("Inputs must lie between zero and one.")

    return X, y, input_columns


# ============================================================
# 2. CHOOSE NUMBER OF CLUSTERS
# ============================================================

def compare_cluster_counts(
    df,
    cluster_range=range(2, 6),
    random_state=42,
):
    """
    Compare KMeans specifications using silhouette score and inertia.
    """
    X, _, _ = extract_function_data(df)

    rows = []

    for k in cluster_range:
        if k >= len(X):
            continue

        model = KMeans(
            n_clusters=k,
            n_init=30,
            random_state=random_state,
        )

        labels = model.fit_predict(X)

        # Silhouette requires at least two populated clusters.
        unique_labels = np.unique(labels)

        if len(unique_labels) < 2:
            silhouette = np.nan
        else:
            silhouette = silhouette_score(
                X,
                labels,
            )

        rows.append({
            "n_clusters": k,
            "silhouette_score": silhouette,
            "inertia": model.inertia_,
        })

    return pd.DataFrame(rows)


def choose_cluster_count(
    df,
    cluster_range=range(2, 6),
    random_state=42,
):
    """
    Choose the number of clusters with the highest silhouette score.
    """
    diagnostics = compare_cluster_counts(
        df=df,
        cluster_range=cluster_range,
        random_state=random_state,
    )

    valid = diagnostics.dropna(
        subset=["silhouette_score"]
    )

    if valid.empty:
        return 2, diagnostics

    best_row = valid.loc[
        valid["silhouette_score"].idxmax()
    ]

    return int(best_row["n_clusters"]), diagnostics


# ============================================================
# 3. FIT CLUSTERS AND CALCULATE SUMMARIES
# ============================================================

def fit_function_clusters(
    df,
    n_clusters=None,
    cluster_range=range(2, 6),
    top_quantile=0.80,
    random_state=42,
):
    """
    Fit input-space KMeans clusters and describe their outputs.

    Clusters are formed using X only. Output is then used to assess
    which clusters perform well.
    """
    X, y, input_columns = extract_function_data(df)

    if n_clusters is None:
        n_clusters, diagnostics = choose_cluster_count(
            df=df,
            cluster_range=cluster_range,
            random_state=random_state,
        )
    else:
        diagnostics = compare_cluster_counts(
            df=df,
            cluster_range=cluster_range,
            random_state=random_state,
        )

    if n_clusters >= len(X):
        raise ValueError(
            "n_clusters must be smaller than the number of observations."
        )

    model = KMeans(
        n_clusters=n_clusters,
        n_init=30,
        random_state=random_state,
    )

    labels = model.fit_predict(X)
    centroids = model.cluster_centers_

    best_idx = int(np.argmax(y))
    best_cluster = int(labels[best_idx])

    top_threshold = np.quantile(
        y,
        top_quantile,
    )

    top_mask = y >= top_threshold

    # Distance of every observation from its assigned centroid.
    assigned_centroids = centroids[labels]

    distance_to_centroid = np.linalg.norm(
        X - assigned_centroids,
        axis=1,
    )

    # Distance from current incumbent.
    distance_to_best = np.linalg.norm(
        X - X[best_idx],
        axis=1,
    )

    labelled_data = df.copy()
    labelled_data["cluster"] = labels
    labelled_data["distance_to_centroid"] = distance_to_centroid
    labelled_data["distance_to_best"] = distance_to_best
    labelled_data["top_observation"] = top_mask

    summary_rows = []

    for cluster_id in range(n_clusters):
        mask = labels == cluster_id

        cluster_X = X[mask]
        cluster_y = y[mask]

        centroid = centroids[cluster_id]

        cluster_spread = np.mean(
            np.linalg.norm(
                cluster_X - centroid,
                axis=1,
            )
        )

        best_local_index = np.argmax(cluster_y)
        best_local_x = cluster_X[best_local_index]

        summary_rows.append({
            "cluster": cluster_id,
            "contains_global_best": cluster_id == best_cluster,
            "n_points": int(mask.sum()),
            "mean_output": float(np.mean(cluster_y)),
            "median_output": float(np.median(cluster_y)),
            "best_output": float(np.max(cluster_y)),
            "worst_output": float(np.min(cluster_y)),
            "output_std": float(np.std(cluster_y)),
            "cluster_spread": float(cluster_spread),
            "centroid": centroid.tolist(),
            "best_point_in_cluster": best_local_x.tolist(),
        })

    cluster_summary = (
        pd.DataFrame(summary_rows)
        .sort_values(
            ["contains_global_best", "best_output"],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )

    if len(np.unique(labels)) > 1:
        silhouette = silhouette_score(
            X,
            labels,
        )
    else:
        silhouette = np.nan

    return {
        "data": labelled_data,
        "X": X,
        "y": y,
        "input_columns": input_columns,
        "labels": labels,
        "centroids": centroids,
        "model": model,
        "n_clusters": n_clusters,
        "silhouette_score": silhouette,
        "cluster_diagnostics": diagnostics,
        "cluster_summary": cluster_summary,
        "best_idx": best_idx,
        "best_x": X[best_idx],
        "best_y": float(y[best_idx]),
        "best_cluster": best_cluster,
        "top_threshold": float(top_threshold),
        "top_mask": top_mask,
    }


# ============================================================
# 4. CLUSTER-COUNT DIAGNOSTIC PLOT
# ============================================================

def plot_cluster_count_diagnostics(
    cluster_result,
    function_name,
):
    diagnostics = cluster_result[
        "cluster_diagnostics"
    ]

    fig, ax = plt.subplots(
        figsize=(7, 4)
    )

    ax.plot(
        diagnostics["n_clusters"],
        diagnostics["silhouette_score"],
        marker="o",
    )

    selected_k = cluster_result["n_clusters"]

    selected_row = diagnostics[
        diagnostics["n_clusters"] == selected_k
    ]

    if not selected_row.empty:
        ax.scatter(
            selected_row["n_clusters"],
            selected_row["silhouette_score"],
            marker="*",
            s=220,
            edgecolors="black",
            label=f"Selected k = {selected_k}",
        )

    ax.set_xlabel("Number of clusters")
    ax.set_ylabel("Silhouette score")
    ax.set_title(
        f"{function_name}: cluster-count comparison"
    )
    ax.set_xticks(
        diagnostics["n_clusters"]
    )
    ax.grid(alpha=0.25)
    ax.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# 5. DIRECT 2D CLUSTER PLOT
# ============================================================

def plot_clusters_2d(
    cluster_result,
    function_name,
    recommended_x=None,
):
    X = cluster_result["X"]
    labels = cluster_result["labels"]
    centroids = cluster_result["centroids"]
    top_mask = cluster_result["top_mask"]
    best_idx = cluster_result["best_idx"]
    input_columns = cluster_result["input_columns"]

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    scatter = ax.scatter(
        X[:, 0],
        X[:, 1],
        c=labels,
        s=65,
        alpha=0.75,
    )

    ax.scatter(
        X[top_mask, 0],
        X[top_mask, 1],
        s=160,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label="Top 20% output",
    )

    ax.scatter(
        centroids[:, 0],
        centroids[:, 1],
        marker="X",
        s=220,
        edgecolors="black",
        linewidths=1.2,
        label="Cluster centroids",
    )

    ax.scatter(
        X[best_idx, 0],
        X[best_idx, 1],
        marker="*",
        s=340,
        edgecolors="black",
        linewidths=1.2,
        label="Best observed",
    )

    if recommended_x is not None:
        recommended_x = np.asarray(
            recommended_x,
            dtype=float,
        )

        if len(recommended_x) != 2:
            raise ValueError(
                f"{function_name} recommendation must have two coordinates."
            )

        ax.scatter(
            recommended_x[0],
            recommended_x[1],
            marker="D",
            s=180,
            edgecolors="black",
            linewidths=1.2,
            label="Proposed query",
        )

        ax.plot(
            [
                X[best_idx, 0],
                recommended_x[0],
            ],
            [
                X[best_idx, 1],
                recommended_x[1],
            ],
            linestyle="--",
            linewidth=1.2,
        )

    ax.set_xlabel(input_columns[0])
    ax.set_ylabel(input_columns[1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.set_title(
        f"{function_name}: input-space clusters"
    )

    ax.grid(alpha=0.20)
    ax.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# 6. DIRECT 3D CLUSTER PLOT
# ============================================================

def plot_clusters_3d(
    cluster_result,
    function_name,
    recommended_x=None,
):
    X = cluster_result["X"]
    labels = cluster_result["labels"]
    centroids = cluster_result["centroids"]
    top_mask = cluster_result["top_mask"]
    best_idx = cluster_result["best_idx"]
    input_columns = cluster_result["input_columns"]

    fig = plt.figure(
        figsize=(9, 7)
    )

    ax = fig.add_subplot(
        111,
        projection="3d",
    )

    ax.scatter(
        X[:, 0],
        X[:, 1],
        X[:, 2],
        c=labels,
        s=55,
        alpha=0.75,
    )

    ax.scatter(
        X[top_mask, 0],
        X[top_mask, 1],
        X[top_mask, 2],
        s=150,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label="Top 20% output",
    )

    ax.scatter(
        centroids[:, 0],
        centroids[:, 1],
        centroids[:, 2],
        marker="X",
        s=220,
        edgecolors="black",
        linewidths=1.2,
        label="Cluster centroids",
    )

    ax.scatter(
        X[best_idx, 0],
        X[best_idx, 1],
        X[best_idx, 2],
        marker="*",
        s=340,
        edgecolors="black",
        linewidths=1.2,
        label="Best observed",
    )

    if recommended_x is not None:
        recommended_x = np.asarray(
            recommended_x,
            dtype=float,
        )

        ax.scatter(
            recommended_x[0],
            recommended_x[1],
            recommended_x[2],
            marker="D",
            s=180,
            edgecolors="black",
            linewidths=1.2,
            label="Proposed query",
        )

    ax.set_xlabel(input_columns[0])
    ax.set_ylabel(input_columns[1])
    ax.set_zlabel(input_columns[2])

    ax.set_title(
        f"{function_name}: 3D input-space clusters"
    )

    ax.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# 7. PCA PROJECTION COLOURED BY CLUSTER
# ============================================================

def plot_clusters_pca(
    cluster_result,
    function_name,
    recommended_x=None,
):
    X = cluster_result["X"]
    labels = cluster_result["labels"]
    centroids = cluster_result["centroids"]
    top_mask = cluster_result["top_mask"]
    best_idx = cluster_result["best_idx"]

    pca = PCA(n_components=2)

    X_pca = pca.fit_transform(X)
    centroids_pca = pca.transform(centroids)

    explained = (
        pca.explained_variance_ratio_
        * 100
    )

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=labels,
        s=65,
        alpha=0.75,
    )

    ax.scatter(
        X_pca[top_mask, 0],
        X_pca[top_mask, 1],
        s=160,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label="Top 20% output",
    )

    ax.scatter(
        centroids_pca[:, 0],
        centroids_pca[:, 1],
        marker="X",
        s=220,
        edgecolors="black",
        linewidths=1.2,
        label="Cluster centroids",
    )

    ax.scatter(
        X_pca[best_idx, 0],
        X_pca[best_idx, 1],
        marker="*",
        s=340,
        edgecolors="black",
        linewidths=1.2,
        label="Best observed",
    )

    recommended_pca = None

    if recommended_x is not None:
        recommended_x = np.asarray(
            recommended_x,
            dtype=float,
        ).reshape(1, -1)

        recommended_pca = pca.transform(
            recommended_x
        )[0]

        ax.scatter(
            recommended_pca[0],
            recommended_pca[1],
            marker="D",
            s=180,
            edgecolors="black",
            linewidths=1.2,
            label="Proposed query",
        )

        ax.plot(
            [
                X_pca[best_idx, 0],
                recommended_pca[0],
            ],
            [
                X_pca[best_idx, 1],
                recommended_pca[1],
            ],
            linestyle="--",
            linewidth=1.2,
        )

    ax.set_xlabel(
        f"PC1 ({explained[0]:.1f}% variance)"
    )
    ax.set_ylabel(
        f"PC2 ({explained[1]:.1f}% variance)"
    )

    ax.set_title(
        f"{function_name}: PCA view of input clusters"
    )

    ax.grid(alpha=0.20)
    ax.legend()

    plt.tight_layout()
    plt.show()

    return {
        "pca": pca,
        "X_pca": X_pca,
        "centroids_pca": centroids_pca,
        "recommended_pca": recommended_pca,
        "explained_variance_percent": explained,
    }


# ============================================================
# 8. PCA PROJECTION COLOURED BY OUTPUT
# ============================================================

def plot_pca_by_output(
    cluster_result,
    function_name,
    recommended_x=None,
):
    X = cluster_result["X"]
    y = cluster_result["y"]
    best_idx = cluster_result["best_idx"]

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    explained = (
        pca.explained_variance_ratio_
        * 100
    )

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    scatter = ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=y,
        s=70,
        alpha=0.80,
    )

    colour_bar = plt.colorbar(
        scatter,
        ax=ax,
    )
    colour_bar.set_label(
        "Observed output"
    )

    ax.scatter(
        X_pca[best_idx, 0],
        X_pca[best_idx, 1],
        marker="*",
        s=340,
        edgecolors="black",
        linewidths=1.2,
        label="Best observed",
    )

    if recommended_x is not None:
        recommended_pca = pca.transform(
            np.asarray(
                recommended_x,
                dtype=float,
            ).reshape(1, -1)
        )[0]

        ax.scatter(
            recommended_pca[0],
            recommended_pca[1],
            marker="D",
            s=180,
            edgecolors="black",
            linewidths=1.2,
            label="Proposed query",
        )

    ax.set_xlabel(
        f"PC1 ({explained[0]:.1f}% variance)"
    )
    ax.set_ylabel(
        f"PC2 ({explained[1]:.1f}% variance)"
    )

    ax.set_title(
        f"{function_name}: PCA coloured by output"
    )

    ax.grid(alpha=0.20)
    ax.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# 9. DISTANCE-TO-BEST PLOT
# ============================================================

def plot_output_vs_distance_to_best(
    cluster_result,
    function_name,
):
    data = cluster_result["data"]

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    scatter = ax.scatter(
        data["distance_to_best"],
        data["output"],
        c=data["cluster"],
        s=65,
        alpha=0.80,
    )

    ax.axhline(
        cluster_result["best_y"],
        linestyle="--",
        linewidth=1,
        label="Best observed output",
    )

    ax.set_xlabel(
        "Euclidean distance from best observed point"
    )
    ax.set_ylabel("Observed output")

    ax.set_title(
        f"{function_name}: output versus distance from incumbent"
    )

    ax.grid(alpha=0.20)
    ax.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# 10. FIND CLUSTER AND DISTANCES FOR A PROPOSED QUERY
# ============================================================

def describe_recommended_query(
    cluster_result,
    recommended_x,
):
    """
    Identify which cluster a proposed query targets and calculate
    distances to the centroid and incumbent.
    """
    recommended_x = np.asarray(
        recommended_x,
        dtype=float,
    ).reshape(1, -1)

    assigned_cluster = int(
        cluster_result["model"].predict(
            recommended_x
        )[0]
    )

    centroid = cluster_result[
        "centroids"
    ][assigned_cluster]

    distance_to_centroid = float(
        np.linalg.norm(
            recommended_x[0] - centroid
        )
    )

    distance_to_best = float(
        np.linalg.norm(
            recommended_x[0]
            - cluster_result["best_x"]
        )
    )

    distances_to_observations = (
        pairwise_distances(
            recommended_x,
            cluster_result["X"],
        )[0]
    )

    nearest_idx = int(
        np.argmin(
            distances_to_observations
        )
    )

    return {
        "target_cluster": assigned_cluster,
        "target_cluster_is_best_cluster":
            assigned_cluster
            == cluster_result["best_cluster"],
        "distance_to_cluster_centroid":
            distance_to_centroid,
        "distance_to_best_observation":
            distance_to_best,
        "distance_to_nearest_observation":
            float(
                distances_to_observations[
                    nearest_idx
                ]
            ),
        "nearest_observation_x":
            cluster_result["X"][
                nearest_idx
            ].tolist(),
        "nearest_observation_output":
            float(
                cluster_result["y"][
                    nearest_idx
                ]
            ),
    }


# ============================================================
# 11. COMPLETE PIPELINE FOR ALL FUNCTIONS
# ============================================================

def run_cluster_visualisation_pipeline(
    function_data,
    recommendations=None,
    manual_cluster_counts=None,
    cluster_range=range(2, 6),
    top_quantile=0.80,
    random_state=42,
    show_diagnostics=True,
    show_distance_plots=True,
):
    """
    Fit, summarise and plot clusters for all functions.

    Parameters
    ----------
    function_data : dict
        Existing dictionary:
            function_data["function_1"] = df

    recommendations : dict or None
        Optional:
            {
                "function_1": [x1, x2],
                ...
            }

    manual_cluster_counts : dict or None
        Optional:
            {
                "function_1": 3,
                "function_2": 2,
                ...
            }

        If omitted, silhouette score chooses k automatically.
    """
    recommendations = (
        {}
        if recommendations is None
        else recommendations
    )

    manual_cluster_counts = (
        {}
        if manual_cluster_counts is None
        else manual_cluster_counts
    )

    cluster_results = {}
    pca_results = {}
    summary_rows = []
    recommendation_rows = []

    for n in range(1, 9):
        function_name = f"function_{n}"

        if function_name not in function_data:
            print(
                f"{function_name} not found; skipping."
            )
            continue

        print("\n" + "=" * 90)
        print(function_name.upper())
        print("=" * 90)

        df = function_data[function_name]

        selected_k = manual_cluster_counts.get(
            function_name
        )

        cluster_result = fit_function_clusters(
            df=df,
            n_clusters=selected_k,
            cluster_range=cluster_range,
            top_quantile=top_quantile,
            random_state=random_state + n,
        )

        cluster_results[
            function_name
        ] = cluster_result

        print(
            f"Selected clusters: "
            f"{cluster_result['n_clusters']}"
        )
        print(
            f"Silhouette score: "
            f"{cluster_result['silhouette_score']:.3f}"
        )
        print(
            f"Best cluster: "
            f"{cluster_result['best_cluster']}"
        )
        print(
            f"Best observed output: "
            f"{cluster_result['best_y']:.6f}"
        )

        display(
            cluster_result[
                "cluster_summary"
            ]
        )

        if show_diagnostics:
            plot_cluster_count_diagnostics(
                cluster_result=cluster_result,
                function_name=f"Function {n}",
            )

        recommended_x = recommendations.get(
            function_name
        )

        dim = cluster_result["X"].shape[1]

        # Direct 2D plots for Functions 1 and 2.
        if dim == 2:
            plot_clusters_2d(
                cluster_result=cluster_result,
                function_name=f"Function {n}",
                recommended_x=recommended_x,
            )

        # Direct 3D plus PCA for Function 3.
        elif dim == 3:
            plot_clusters_3d(
                cluster_result=cluster_result,
                function_name=f"Function {n}",
                recommended_x=recommended_x,
            )

            pca_results[
                function_name
            ] = plot_clusters_pca(
                cluster_result=cluster_result,
                function_name=f"Function {n}",
                recommended_x=recommended_x,
            )

        # PCA views for Functions 4–8.
        else:
            pca_results[
                function_name
            ] = plot_clusters_pca(
                cluster_result=cluster_result,
                function_name=f"Function {n}",
                recommended_x=recommended_x,
            )

        # Output-coloured PCA for all dimensions above 2.
        if dim > 2:
            plot_pca_by_output(
                cluster_result=cluster_result,
                function_name=f"Function {n}",
                recommended_x=recommended_x,
            )

        if show_distance_plots:
            plot_output_vs_distance_to_best(
                cluster_result=cluster_result,
                function_name=f"Function {n}",
            )

        summary_rows.append({
            "function": function_name,
            "dimensions": dim,
            "n_observations": len(df),
            "n_clusters":
                cluster_result["n_clusters"],
            "silhouette_score":
                cluster_result[
                    "silhouette_score"
                ],
            "best_cluster":
                cluster_result[
                    "best_cluster"
                ],
            "best_observed_y":
                cluster_result["best_y"],
            "best_observed_x":
                cluster_result[
                    "best_x"
                ].tolist(),
        })

        if recommended_x is not None:
            recommendation_description = (
                describe_recommended_query(
                    cluster_result=cluster_result,
                    recommended_x=recommended_x,
                )
            )

            recommendation_description[
                "function"
            ] = function_name

            recommendation_description[
                "recommended_x"
            ] = recommended_x

            recommendation_rows.append(
                recommendation_description
            )

    overall_summary = pd.DataFrame(
        summary_rows
    )

    recommendation_summary = pd.DataFrame(
        recommendation_rows
    )

    return {
        "cluster_results": cluster_results,
        "pca_results": pca_results,
        "overall_summary": overall_summary,
        "recommendation_summary":
            recommendation_summary,
    }